# Masar Modern Data Engineering
## Complete Labs

### Local Runtime Compatibility Setup

Purpose: Prepare the local machine before any Spark session starts.

This cell:

- Forces the notebook to use Java 17, which is required by the course runtime.

- Removes the old standalone Spark path so the virtual environment controls the Spark version.

- Adds the required Delta Lake and Kafka connector packages to the Spark JVM classpath.

- Prints the resolved Java and Spark-related environment values for verification.

- This setup must run before the first Spark session because JVM dependencies are fixed when Spark starts.

In [1]:
import os
import subprocess

# ============================================================
# Java 17
# ============================================================

JAVA17 = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

os.environ["JAVA_HOME"] = JAVA17


# ============================================================
# Do not use the old standalone Spark 3.5.0 installation
# ============================================================

os.environ.pop("SPARK_HOME", None)

OLD_SPARK = "/Users/nouraalmadhi/spark/spark-3.5.0-bin-hadoop3"

path_parts = [
    p
    for p in os.environ.get("PATH", "").split(":")
    if OLD_SPARK not in p
]

os.environ["PATH"] = (
    JAVA17
    + "/bin:"
    + ":".join(path_parts)
)


# ============================================================
# JVM packages needed by the COMPLETE merged notebook
#
# Delta Lake  -> used from Day 1 onward
# Kafka       -> needed later for the streaming lab
# ============================================================

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.3.3,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 "
    "pyspark-shell"
)


# ============================================================
# Quick verification
# ============================================================

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYSPARK_SUBMIT_ARGS:", os.environ["PYSPARK_SUBMIT_ARGS"])

java_result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(java_result.stderr)

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
SPARK_HOME: None
PYSPARK_SUBMIT_ARGS: --packages io.delta:delta-spark_2.12:3.3.3,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 pyspark-shell
openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment Homebrew (build 17.0.20.1+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.20.1+0, mixed mode, sharing)



# DAY01

## Environment and Repository Setup

Purpose: Make sure the notebook is running inside the complete project repository with the expected runtime.

This cell:

- Detects whether execution is happening in Colab or locally.

- Locates or clones the project repository when needed.

- Checks the pinned PySpark, Delta Lake, and Py4J versions.

- Configures Java 17.

- Resolves the project root and prints the active Python version.

No project data is transformed in this step; it only prepares the execution environment.

In [2]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
IS_COLAB = Path('/content').is_dir() and 'google.colab' in sys.modules
if IS_COLAB:
    ROOT = Path('/content/masar-modern-data-engineering')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/almiyead-rgb/masar-modern-data-engineering.git',str(ROOT)], check=True)
    pins = {'pyspark':'3.5.8','delta-spark':'3.3.3','py4j':'0.10.9.9'}
    missing = []
    for package, version in pins.items():
        try: observed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: observed = None
        if observed != version: missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
    candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    if not any((p/'bin/java').is_file() for p in candidates):
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
        candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    os.environ['JAVA_HOME'] = str(next(p for p in candidates if (p/'bin/java').is_file()))
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.json').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Open from the repository root; see docs/SETUP.md')
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


Repository: /Users/nouraalmadhi/Desktop/Masar_Data_Engineering
Python: 3.11.16


## Create an Isolated Source-Inspection Run

Purpose: Prepare a clean location for source-inspection evidence.

This cell:

- Locates the repository root using course.json.

- Adds src/ to the Python import path so the local masar package can be imported.

- Creates the outputs/ directory when necessary.

- Creates a temporary Day 1 run directory for inspection results.

The source files are not modified.

In [3]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


## Verify and Load the Source Feeds

Purpose: Confirm that the supplied dataset is intact before processing it.

This cell:

- Points to the fixed Masar dataset.

- Validates the source manifest and file hashes.

- Loads the trips, drivers, and GPS feeds.

- Prints the dataset label, row counts, trip columns, and one example GPS event.

This establishes the trusted input state before Bronze ingestion.

In [4]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


## Profile the Raw Sources

Purpose: Understand the source data before applying any cleaning or transformation.

This cell profiles:

- Column-level characteristics and source counts.

- Relationships between the supplied feeds.

- City-label variations that may later require conformance.

The result is observational only; no source values are rewritten.

In [5]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

## Validate and Save Source-Inspection Evidence

Purpose: Turn the inspection results into reproducible evidence.

This cell:

- Verifies that every source-level check passed.

- Stops with an assertion if any required check fails.

- Saves the complete inspection result as JSON.

- Prints the final check status and output location.

The saved report documents the state of the source data before Bronze is created.

In [6]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Source inspection complete. Continue to the Bronze section.")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Source inspection complete. Continue to the Bronze section.


##  Inspect the Spark Runtime

Purpose: Verify the actual local runtime before creating Delta tables.

This cell:

- Resolves the repository and source dataset again for the Bronze section.

- Imports the course runtime helpers.

- Prints the detected Python, Java, Spark, Delta, and related environment information.

It checks the environment without starting Spark.

In [7]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.16",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


## Start a New Bronze Workspace

Purpose: Create an isolated workspace for the Bronze ingestion run.

This cell:

- Enforces the required runtime versions.

- Verifies that the fixed dataset is present.

- Creates a new project workspace for the Bronze tables and reports.

All Bronze outputs produced by the next cells are written under this workspace.

In [8]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_jnzj35lx


## Load Bronze Ingestion Functions

Purpose: Import the functions used to read, ingest, and validate the Bronze layer.

The imported functions separate three responsibilities:

- raw_frame reads a source feed in its raw form.

- ingest_feed appends a feed to Bronze with ingestion metadata.

- verify_bronze checks that the resulting Delta tables satisfy the lab contract.

Spark is not started in this cell.

In [9]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze functions loaded. The next cell writes and reads the Delta tables.")

Bronze functions loaded. The next cell writes and reads the Delta tables.


## Ingest, Replay, and Validate Bronze

Purpose: Build the append-only Bronze layer and prove that source replays preserve history.

This cell:

- Starts Spark with Delta support.

- Reads and previews raw trip data.

- Ingests trips, drivers, and GPS events into Bronze.

- Replays the trips feed using a second batch identifier.

- Verifies Bronze counts, lineage metadata, and Delta-table behaviour.

- Records the successful Bronze workspace for later labs.

- Stops Spark in finally so the session closes even if the lab fails.

The replay is intentional: Bronze keeps delivery history, while deduplication is handled later in Silver.

In [10]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1
Ivy Default Cache set to: /Users/nouraalmadhi/.ivy2/cache
The jars for the packages stored in: /Users/nouraalmadhi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2b03c3a9-14b0-499e-b269-a989011217a9;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.8 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.8 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in cent

:: loading settings :: url = jar:file:/Users/nouraalmadhi/Desktop/Masar_Data_Engineering/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/09/15 11:12:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark: 3.5.8
root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- start_ts: string (nullable = true)
 |-- end_ts: string (nullable = true)
 |-- fare_sar: string (nullable = true)
 |-- distance_km: string (nullable = true)

+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|trip_id  |driver_id|city  |start_ts                 |end_ts                   |fare_sar|distance_km|
+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|SYN_T0001|SYN_D001 |Riyadh|2026-06-01T06:00:00+03:00|2026-06-01T06:08:00+03:00|18.00   |3.50       |
|SYN_T0002|SYN_D002 |Riyadh|2026-06-01T08:00:00+03:00|2026-06-01T08:11:00+03:00|19.25   |3.85       |
|SYN_T0003|SYN_D001 |Riyadh|2026-06-01T10:00:00+03:00|2026-06-01T10:14:00+03:00|20.50   |4.20       |
+---------+---------+------+-------------------------+-------------------------+--------+---

## Create a Cost-Model Evidence Run

Purpose: Start a separate evidence directory for the compute-cost exercise.

This cell:

- Resolves the repository root.

- Adds the project source package to Python imports.

- Creates a temporary Day 1 output directory for cost-model evidence.

This section is separate from the Bronze data-processing workspace.

In [11]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


## Inspect the Cost Assumptions

Purpose: Make the cost model transparent before comparing operating strategies.

This cell:

- Loads the default teaching assumptions.

- Runs the base cost model.

- Prints both the assumptions and the resulting base scenario.

The values are teaching units used to compare strategies, not a claim about an actual cloud bill.

In [12]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

{
  "days": "30",
  "hours_per_day": "24",
  "cores": "4",
  "price_per_core_hour": "0.50",
  "storage_gib": "100",
  "storage_price_per_gib_month": "0.20",
  "work_hours_per_day": "2",
  "startup_hours_per_day": "0.25",
  "scheduled_extra_monthly": "30",
  "always_on_extra_monthly": "0"
}
{
  "unit": "TU (hypothetical teaching units; not currency)",
  "always_on_hours": "720",
  "scheduled_hours": "67.50",
  "always_on_compute": "1440.00",
  "scheduled_compute": "135.0000",
  "storage_each": "20.00",
  "always_on_total": "1460.00",
  "scheduled_total": "185.0000",
  "difference": "1275.0000",
  "reduction_fraction": "0.8732876712328767123287671233",
  "break_even_work_hours_per_day": "23.25"
}


## Evaluate Workload Sensitivity

Purpose: Show how the cost comparison changes as daily compute usage increases.

This cell iterates through the predefined sensitivity scenarios and prints:

- Work hours per day.

- Always-on compute cost.

- Scheduled compute cost.

- The difference between the two.

Only workload duration changes, making the comparison easier to interpret.

In [13]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

Work h/day | Always-on TU | Scheduled TU | Difference TU
         0 |      1460.00 |        50.00 |       1410.00
         1 |      1460.00 |     125.0000 |     1335.0000
         2 |      1460.00 |     185.0000 |     1275.0000
         4 |      1460.00 |     305.0000 |     1155.0000
         8 |      1460.00 |     545.0000 |      915.0000
        12 |      1460.00 |     785.0000 |      675.0000
        18 |      1460.00 |    1145.0000 |      315.0000
        23 |      1460.00 |    1445.0000 |       15.0000
     23.25 |      1460.00 |    1460.0000 |        0.0000
     23.75 |      1460.00 |    1490.0000 |      -30.0000


## Test Cost-Model Boundaries

Purpose: Verify that the arithmetic behaves correctly at important edge cases.

This cell checks:

- The expected base totals.

- The break-even workload.

- A counterexample where scheduled compute is no longer cheaper.

- A zero-price edge case.

- Rejection of an invalid negative input.

An assertion fails if any of these checks do not behave as expected.

In [14]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

{
  "base_totals": true,
  "break_even": true,
  "counterexample": true,
  "zero_rate": true,
  "negative_input_rejected": true
}


## Save Cost-Model Evidence

Purpose: Persist the cost-model results for review and submission.

This cell writes the complete result and checks to a JSON file and prints the saved artifact name.

The report keeps the cost exercise reproducible without mixing it with the later measured Spark benchmark.

In [15]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Cost arithmetic complete. Continue to the measured Spark comparison.")

PASS: hypothetical cost arithmetic only
Saved: cost_model_result.json
Cost arithmetic complete. Continue to the measured Spark comparison.


## Recheck the Runtime Before Benchmarking

Purpose: Verify the environment before running the measured Spark comparison.

This cell:

- Resolves the repository and source path.

- Loads the runtime helpers.

- Prints the currently detected environment.

No benchmark is executed yet.

In [16]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.16",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


## Reuse the Successful Bronze Workspace

Purpose: Benchmark the project state created by Lab 01 instead of rebuilding the data.

This cell:

- Enforces the required environment.

- Locates the recorded successful Bronze workspace.

- Verifies the fixed source dataset.

- Starts Spark using the existing project state.

This preserves the cumulative nature of the project.

In [17]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_jnzj35lx


## Run the Spark Scan Benchmark

Purpose: Compare equivalent source and Delta reads using measured Spark evidence.

This cell:

- Runs the benchmark several times.

- Verifies that the compared operations return the same business result.

- Prints observed measurements and query-plan locations.

- Saves the benchmark evidence under the existing workspace.

- Stops Spark after the measurements complete.

The small dataset makes these timings educational observations rather than production performance claims.

In [18]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      0.03346883300400805,
      0.035255957998742815,
      0.03379783300624695,
      0.034085999999661
    ],
    "median_s": 0.03394191650295397,
    "min_s": 0.03346883300400805,
    "max_s": 0.035255957998742815
  },
  "delta_v0": {
    "samples_s": [
      0.19756320799933746,
      0.19101970799965784,
      0.1892827500050771,
      0.19458279099490028
    ],
    "median_s": 0.19280124949727906,
    "min_s": 0.1892827500050771,
    "max_s": 0.19756320799933746
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_jnzj35lx/reports/benchmark.json


## Package the Handoff

Purpose: Preserve Day 1 state so the next day can continue from the same Bronze workspace.

This cell:

- Locates the successful Bronze workspace pointer.

- Collects the Delta files and small JSON reports.

- Creates day01_handoff.zip.

- Validates that the ZIP is readable and contains the required evidence.

- Downloads the archive automatically when running in Colab.

The archive transfers workspace state; the executed notebook and notes remain the human-readable evidence.


In [19]:
# DAY01_HANDOFF_V2: retain the pointer and all small reports as well as Delta files.
from pathlib import Path
import zipfile
from masar.workspace import completed_bronze_workspace
WORK = completed_bronze_workspace(ROOT)
pointer = ROOT / 'outputs/day01_bronze_success.json'
files_to_save = {pointer, *(p for p in WORK.rglob('*') if p.is_file())}
for name in ('source_inspection.json', 'cost_model_result.json'):
    files_to_save.update((ROOT / 'outputs').rglob(name))
archive = ROOT / 'outputs/day01_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files_to_save):
        bundle.write(path, arcname=path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert bundle.read('outputs/day01_bronze_success.json') == pointer.read_bytes()
    assert any(name.endswith('source_inspection.json') for name in bundle.namelist())
    assert any(name.endswith('cost_model_result.json') for name in bundle.namelist())
print('Keep this ZIP for the next day:', archive)
print('Also save this notebook with outputs and your LAB01/LAB02 notes.')
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))

Keep this ZIP for the next day: /Users/nouraalmadhi/Desktop/Masar_Data_Engineering/outputs/day01_handoff.zip
Also save this notebook with outputs and your LAB01/LAB02 notes.


# DAY02

## Continue from the Existing Bronze Workspace

Purpose: Reuse Day 1 outputs and prepare the environment for the Silver pipeline.

This cell:

- Resolves the repository and fixed source dataset.

- Imports the workspace, runtime, and validation helpers.

- Verifies the dataset and runtime.

- Loads the previously recorded successful Bronze workspace.

Day 2 intentionally continues the same project state instead of rebuilding Bronze.


In [20]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jnzj35lx


## Build Typed Staging Tables

Purpose: Convert Bronze data into typed, normalized staging tables before publishing Silver.

This cell:

- Starts Spark.

- Runs the staging pipeline.

- Validates the staging-stage contract.

- Prints the stage scope, checks, and observed row counts.

- Reads the generated Delta staging table and previews selected trip fields.

- Stops Spark after completion.

Staging is where raw strings begin to become consistently typed and conformed data.


In [21]:
from masar.silver import run_staging_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_staging_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03a_staging', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Observed staging row counts:', result['counts'])
    preview = WORK / ('mini_lakehouse/staging/day02_' + result['run_id'] + '/stg_trips')
    spark.read.format('delta').load(str(preview)).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()


{
  "scope": "DAY02_STAGING_ENGINE",
  "checks": {
    "counts_verified": true,
    "typed_values_match_source_oracle": true,
    "drivers_unique_and_join_safe": true,
    "gps_valid": true,
    "delta_readback": true
  }
}
Observed staging row counts: {'stg_trips': 144, 'stg_drivers': 6, 'stg_gps': 216}
+---------+------+--------+
|trip_id  |city  |fare_sar|
+---------+------+--------+
|SYN_T0001|Riyadh|18.00   |
|SYN_T0001|Riyadh|18.00   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0003|Riyadh|20.50   |
+---------+------+--------+
only showing top 5 rows



## Build Incremental Silver

Purpose: Produce the canonical trusted trip table from Bronze and staging data.

This cell:

- Runs the incremental Silver pipeline.

- Applies the lab's deduplication, late-arrival, and replay scenarios.

- Validates the Silver result.

- Reads the final silver/trips Delta table and previews trusted rows.

- Stops Spark after the run.

The key property demonstrated here is idempotency: rerunning the same logical input must not create duplicate business records.


In [22]:
from masar.silver import run_incremental_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_incremental_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03b_silver', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    spark.read.format('delta').load(str(WORK/'mini_lakehouse/silver/trips')).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()


{
  "scope": "DAY02_SILVER_ENGINE",
  "checks": {
    "all_scenarios_match_independent_oracle": true,
    "business_keys_unique": true,
    "replay_preserves_business_content": true,
    "late_rows_retained": true,
    "actual_delta_files": true
  }
}
+-----------+------+--------+
|trip_id    |city  |fare_sar|
+-----------+------+--------+
|SYN_LATE001|Riyadh|25.00   |
|SYN_LATE002|Jeddah|27.00   |
|SYN_LATE003|Dammam|29.00   |
|SYN_T0001  |Riyadh|18.00   |
|SYN_T0002  |Riyadh|19.25   |
+-----------+------+--------+
only showing top 5 rows



## Validate the dbt Transformation Layer

Purpose: Run the dbt portion of the Silver exercise and verify its staged scenarios.

This cell:

- Executes the dbt lab against isolated copies of the course data.

- Verifies that the dbt workflow completed successfully.

- Prints row counts and total fare for each phase.

- Prints catalog/documentation evidence from the final dbt command.

The phases show that reruns preserve results while late data changes the trusted content only when expected.


In [23]:
from masar.dbt_lab import run_dbt_lab
dbt_report, dbt_path = run_dbt_lab(ROOT)
print(json.dumps({'status': dbt_report['status'], 'phases_completed': len(dbt_report['phases']), 'report': str(dbt_path.relative_to(ROOT)), 'error': dbt_report.get('error')}, indent=2))
assert dbt_report['status'] == 'PASSED_DBT_NATIVE', dbt_report.get('error')

# Observed learning output
for phase in dbt_report['phases']:
    print(phase['phase'], 'rows:', phase['rows'], 'fare SAR:', phase['total_fare_sar'])
print('Catalog evidence:', dbt_report['commands'][-1])


{
  "status": "PASSED_DBT_NATIVE",
  "phases_completed": 4,
  "report": "outputs/dbt_validation_xevqdu2j/reports/dbt_attempt.json",
  "error": null
}
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
Catalog evidence: {'catalog_sha256': '62d29fa7ef20302f06c287ad99659f65392f10b86f27b8085e492ab651512c03', 'models_documented': 6, 'sources_documented': 3, 'metadata_method': 'native DESCRIBE TABLE EXTENDED', 'phase': 'documentation', 'command': ['docs', 'generate'], 'target': 'dbt/commands/documentation/target'}


## Package the Handoff

Purpose: Preserve the Silver workspace and dbt evidence for the next stage.

This cell:

- Creates day02_handoff.zip.

- Includes the previous Bronze-success pointer.

- Includes dbt validation artifacts.

- Includes the current workspace files.

- Tests the resulting ZIP before reporting success.

The next day can therefore continue from the same cumulative project state.


In [24]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day02_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    dbt_workspace = dbt_path.parent.parent
    for p in sorted(dbt_workspace.rglob('*')):
        if p.is_file():
            bundle.write(p, p.relative_to(ROOT).as_posix())
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day02_handoff.zip


# DAY03

## Resume the Trusted Silver Workspace

Purpose: Prepare the existing project state for Delta reliability exercises.

This cell:

- Resolves the repository and source paths.

- Verifies the source dataset and runtime.

- Loads the recorded project workspace.

- Imports the common stage-result validator.

The following labs operate on the trusted tables created earlier rather than generating a separate dataset.


In [25]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jnzj35lx


## Test Transactions, Corrections, and Version History

Purpose: Demonstrate that Delta tables can change safely while keeping an auditable history.

This cell:

- Runs the transaction and correction scenarios.

- Validates the Lab 04A result contract.

- Prints the table state before correction, after correction, and from a previous Delta version.

- Stops Spark after the exercise.

This stage demonstrates schema/constraint enforcement, deterministic corrections, and time-travel reads.


In [26]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
finally:
    spark.stop()


26/09/15 11:13:53 ERROR Utils: Aborting task
org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint fare_nonnegative ((fare_sar IS NOT NULL) AND (fare_sar >= 0)) violated by row with values:
 - fare_sar : -5.00
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:78)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:106)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:117)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.CheckDeltaInvariant_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unkno

{
  "scope": "DAY03_TRANSACTIONS_ENGINE",
  "checks": {
    "native_correction_matches_source_expectation": true,
    "business_rows_stay_75": true,
    "replay_and_stale_delivery_preserve_values": true,
    "same_revision_conflict_rejected": true,
    "actual_prior_version_read": true,
    "mixed_valid_invalid_batch_rejected_atomically": true
  }
}
{
  "before": {
    "rows": 75,
    "business_digest": "0d16e2795620ae0c0f54d3fcd52a5b13fb2c2a47cd4d0c42c8ddb195f19bc6e0",
    "version": 1,
    "schema": [
      [
        "trip_id",
        "string"
      ],
      [
        "driver_id",
        "string"
      ],
      [
        "city",
        "string"
      ],
      [
        "start_utc",
        "timestamp"
      ],
      [
        "end_utc",
        "timestamp"
      ],
      [
        "trip_date_local",
        "date"
      ],
      [
        "fare_sar",
        "decimal(12,2)"
      ],
      [
        "distance_km",
        "decimal(12,2)"
      ],
      [
        "duration_seconds",

## Test Safe Maintenance and Recovery

Purpose: Exercise Delta maintenance operations without losing the ability to reason about table state.

This cell:

- Runs the isolated maintenance scenarios.

- Validates the maintenance-stage result.

- Prints recovery information and VACUUM evidence.

- Stops Spark after completion.

Maintenance operations are tested on controlled training copies so the main trusted project state remains safe.


In [27]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()


{
  "scope": "DAY03_MAINTENANCE_ENGINE",
  "checks": {
    "unexpected_column_rejected": true,
    "approved_evolution_preserves_business_values": true,
    "compaction_preserves_values": true,
    "delete_affects_copy_only": true,
    "restore_creates_new_commit": true,
    "vacuum_is_non_destructive_dry_run": true,
    "trusted_silver_unchanged": true
  }
}
{
  "recovery": {
    "before": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 0,
      "schema": [
        [
          "trip_id",
          "string"
        ],
        [
          "driver_id",
          "string"
        ],
        [
          "city",
          "string"
        ],
        [
          "start_utc",
          "timestamp"
        ],
        [
          "end_utc",
          "timestamp"
        ],
        [
          "trip_date_local",
          "date"
        ],
        [
          "fare_sar",
          "decimal(12,2)"
        ],
       

## Package the Handoff

Purpose: Preserve the updated Delta workspace for Day 4.

This cell:

- Creates day03_handoff.zip.

- Adds the successful Bronze pointer and all current workspace files.

- Verifies that the archive is not corrupted.

This allows the streaming and quality labs to continue from the same trusted lineage.


In [28]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day03_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day03_handoff.zip


# DAY04

## Prepare the Existing Project for Streaming and Quality

Purpose: Reopen the cumulative workspace before adding continuous ingestion and the quality gate.

This cell:

- Resolves the repository and source dataset.

- Verifies the runtime.

- Loads the existing successful workspace.

- Imports the shared stage validator and Spark launcher.

Kafka is required only when the streaming lab starts.


In [29]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jnzj35lx


## Stream GPS Events from Kafka

Purpose: Demonstrate restart-safe event ingestion using Kafka, Spark Structured Streaming, checkpoints, and Delta.

This cell:

- Starts Spark with Kafka support.

- Runs the streaming scenarios.

- Validates the streaming-stage result.

- Prints transport-row and unique-event counts across phases.

- Reads the resulting Delta event table and previews event IDs, trip IDs, and event timestamps.

- Stops Spark after the streaming exercise.

The phase comparison is used to verify that restart and replay scenarios do not silently corrupt the trusted event set.


In [30]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
    spark.read.format('delta').load(str(WORK/result['event_table'])).select('event_id','trip_id','event_ts').orderBy('event_id').show(5, truncate=False)
finally:
    spark.stop()


{
  "scope": "DAY04_NATIVE_STREAMING",
  "checks": {
    "phase_transport_counts": true,
    "phase_event_counts": true,
    "transport_keys_always_unique": true,
    "restart_same_query_identity": true,
    "restart_new_execution_ids": true,
    "checkpoint_same_for_all_phases": true,
    "actual_checkpoint_files_present": true,
    "producer_consumer_offsets_reconcile": true,
    "source_json_text_preserved": true,
    "event_content_matches_source": true,
    "unique_events_delta_readback": true,
    "all_events_link_to_trusted_trips": true,
    "late_event_retained": true
  }
}
Transport rows: [216, 216, 218, 219]
Unique event IDs: [216, 216, 216, 217]
+-----------+---------+-------------------------+
|event_id   |trip_id  |event_ts                 |
+-----------+---------+-------------------------+
|SYN_E0001_0|SYN_T0001|2026-06-01T06:00:00+03:00|
|SYN_E0001_1|SYN_T0001|2026-06-01T06:04:00+03:00|
|SYN_E0001_2|SYN_T0001|2026-06-01T06:08:00+03:00|
|SYN_E0002_0|SYN_T0002|2026-06-01T0

## Validate, Block, and Quarantine Bad Data

Purpose: Apply an explicit quality gate before data is promoted to trusted outputs.

This cell:

- Runs the Great Expectations-based quality scenarios.

- Validates the quality-stage contract.

- Reads and displays quarantined records with their failure context.

- Counts the rows approved for promotion.

- Stops Spark when finished.

The lab distinguishes between failures that should block promotion entirely and row-level defects that can be quarantined for investigation.


In [31]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Quarantined records:')
    spark.read.format('delta').load(str(WORK/result['quarantine_table'])).show(7, truncate=False)
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()


/Users/nouraalmadhi/Desktop/Masar_Data_Engineering/.venv/lib/python3.11/site-packages/great_expectations/data_context/store/_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(


Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

/Users/nouraalmadhi/Desktop/Masar_Data_Engineering/.venv/lib/python3.11/site-packages/great_expectations/data_context/store/_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(


Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

/Users/nouraalmadhi/Desktop/Masar_Data_Engineering/.venv/lib/python3.11/site-packages/great_expectations/data_context/store/_store_backend.py:88: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(


Calculating Metrics:   0%|          | 0/128 [00:00<?, ?it/s]

{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "trusted_and_rechecked_pass_gx": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "failed_candidate_not_promoted": true,
    "quarantine_delta_readback": true,
    "approved_readback_same_business_contents": true,
    "source_silver_untouched": true,
    "data_docs_exist_for_all_three_cases": true
  }
}
Quarantined records:
+-------------+----------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------------------------------------------------------------+
|candidate_row|trip_id   |reason_codes       |raw_business_json                             

## Package the Handoff

Purpose: Preserve the streaming and quality-controlled project state for final integration.

This cell:

- Creates day04_handoff.zip.

- Includes the project workspace and Bronze-success pointer.

- Validates the generated archive.

The final day consumes this accumulated state rather than recreating earlier outputs.


In [32]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day04_handoff.zip


# DAY05

## Resume the Complete Project State

Purpose: Prepare the cumulative lakehouse for final Gold integration and serving.

This cell:

- Resolves the repository and fixed source dataset.

- Verifies the required runtime.

- Loads the existing project workspace.

- Imports the shared result validator.

The next cells connect the outputs of the previous labs into the final serving workflow.


In [33]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jnzj35lx


## Build Gold and Test Recovery

Purpose: Prove that the integrated Bronze → Silver → Gold flow fails safely when an upstream condition is invalid.

This cell:

- Runs the Gold recovery exercise.

- Validates the Lab 07 stage contract.

- Prints the resulting scope and safety checks.

- Stops Spark after completion.

The exercise focuses on dependency ordering and preventing unsafe downstream publication.


In [34]:
from masar.serving import run_recovery_exercise
spark = start_spark(WORK, kafka=False)
try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result('lab07_gold_recovery', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
finally:
    spark.stop()


{
  "scope": "DAY05_NATIVE_RECOVERY",
  "checks": {
    "injected_failure_observed": true,
    "previous_release_preserved": true,
    "rebuild_has_new_identity": true,
    "content_equal": true
  }
}


## Serve BI and AI Outputs

Purpose: Produce final reporting and feature outputs from the same trusted lineage.

This cell:

- Runs the serving workflow.

- Validates the final serving-stage contract.

- Prints BI reconciliation totals.

- Reads the committed release.

- Displays an example AI feature row and its future-label counterpart.

- Stops Spark after completion.

The final checks confirm that BI and AI products are derived consistently while keeping feature values point-in-time correct.


In [35]:
from masar.serving import run_serving_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result('lab08_serving', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('BI totals:', json.dumps(result['bi_summary'], indent=2))
    from masar.serving import read_release
    _, observed_tables = read_release(spark, WORK)
    print('AI feature example:', observed_tables['ai.zone_hourly_features'][0])
    print('Future label example:', observed_tables['ai.zone_hourly_labels'][0])
finally:
    spark.stop()


{
  "scope": "DAY05_NATIVE_SERVING",
  "checks": {
    "gold.zone_hourly_demand_schema_and_keys": true,
    "gold.driver_daily_schema_and_keys": true,
    "bi.dim_zone_schema_and_keys": true,
    "bi.dim_driver_schema_and_keys": true,
    "bi.dim_date_schema_and_keys": true,
    "bi.fact_trips_schema_and_keys": true,
    "ai.zone_hourly_features_schema_and_keys": true,
    "ai.zone_hourly_labels_schema_and_keys": true,
    "fact_grain_75": true,
    "foreign_keys_valid": true,
    "gold_fact_totals_match": true,
    "events_aggregated_before_join": true,
    "group_grains_reconcile": true,
    "labels_not_fabricated": true,
    "feature_availability_checked": true,
    "feature_label_keys_aligned": true
  }
}
BI totals: [
  {
    "zone_key": "Z_DAMMAM",
    "trip_count": 25,
    "total_fare_sar": "670.40"
  },
  {
    "zone_key": "Z_JEDDAH",
    "trip_count": 25,
    "total_fare_sar": "625.20"
  },
  {
    "zone_key": "Z_RIYADH",
    "trip_count": 25,
    "total_fare_sar": "585.00"
  }

## Package the Final Handoff

Purpose: Preserve the completed cumulative lakehouse state.

This cell:

- Creates day05_handoff.zip.

- Includes the workspace and successful Bronze pointer.

- Validates the archive before reporting success.

The ZIP is a workspace handoff artifact; the executed notebooks, reports, notes, README, decisions, benchmarks, and governance files remain the primary repository evidence.


In [36]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day05_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day05_handoff.zip
